In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats
from pathlib import Path
import sys
import os
from datetime import datetime
from dataScraper import *

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import nameDict


pd.set_option('display.max_columns', None)

In [2]:
today = datetime.today().strftime('%Y%m%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

us_file = get_latest_file('NBA_US_*.csv')
dfs_file = get_latest_file('NBA_DFS_*.csv')

if us_file is None:
    raise ValueError("No US file found")

if dfs_file is None:
    raise ValueError("No DFS file found")

us_df = pd.read_csv(us_file)
lines_dfs = pd.read_csv(dfs_file)

lines_us = us_df[us_df['CATEGORY'] == 'player_points'].copy()


print("US file:", us_file.name)
print("DFS file:", dfs_file.name)
print("DFS latest pull:", lines_dfs['DATA_PULLED_AT'].max())
print("US latest pull:", us_df['DATA_PULLED_AT'].max())

US file: NBA_US_20260328_135457.csv
DFS file: NBA_DFS_20260328_135553.csv
DFS latest pull: 2026-03-28 13:55:53
US latest pull: 2026-03-28 13:54:57


In [3]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')

if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260328_135553.json


,home_team,away_team,commence_time,bookmakers
0,Milwaukee Bucks,San Antonio Spurs,2026-03-28 19:11:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Minnesota Timberwolves,Detroit Pistons,2026-03-28 21:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Charlotte Hornets,Philadelphia 76ers,2026-03-28 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Atlanta Hawks,Sacramento Kings,2026-03-28 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Memphis Grizzlies,Chicago Bulls,2026-03-29 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [9]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,Position
20737,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PF
21122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PG
21371,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C
22243,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

In [10]:
# Change line_bookmaker to e.g. 'PrizePicks' to use that DFS book's lines from lines_dfs
final, tier1_all, final = generalized_best_bets(
    lines_dfs, base_df, us_df, team_dds, nameDict,
    line_bookmaker='Underdog',
)

# Player `POSITION` on each row is set inside generalized_best_bets:
#   merged['POSITION'] = merged['Position']   # latest S26 row per PLAYER_ID → correct per player
# Do not use merged['POSITION'] = df['Position'].iloc[0] — that would assign every player the same slot.

print('Total bets across categories:', len(final))
print('Tier 1 bets:', len(tier1_all))

if not final.empty:
    display(tier1_all.head(20))

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/points_model/dataScraper.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/points_model/dataScraper.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/points_mo

Total bets across categories: 162
Tier 1 bets: 48


/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/points_model/dataScraper.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/points_model/dataScraper.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(


,PLAYER_NAME,POSITION,TEAM_NAME,OPPONENT,HOME_AWAY,TEAM_SPREAD,GAME_TOTAL,CATEGORY,LINE,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES,MATCHUP_EDGE,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,TOTAL_BOOST,IS_UNDERDOG,BET_FLAG,COMMENCE_TIME
0,Joel Embiid,C,Philadelphia 76ers,Charlotte Hornets,AWAY,6.5,230.5,player_points,26.5,120,-160,0.455,0.615,32.1,32.5,27.00,2.0,0.50,5.47,5.6,6.0,-1.024,0.847,0.153,86.34,-75.14,0.6,0.8,0.73,0.56,34.73,5.58,0.35,0.05,1.05,1,True,2026-03-28
1,Tyrese Maxey,PG,Philadelphia 76ers,Charlotte Hornets,AWAY,6.5,230.5,player_points,24.5,-110,-105,0.524,0.512,29.6,29.5,23.60,5.0,-0.90,4.95,5.1,5.0,-1.030,0.848,0.152,61.89,-70.32,0.8,0.9,0.87,0.70,35.82,4.86,0.33,0.04,1.05,1,True,2026-03-28
2,Nickeil Alexander-Walker,SG,Atlanta Hawks,Sacramento Kings,HOME,-14.5,236.5,player_points,20.5,102,-115,0.495,0.535,24.1,22.5,10.40,5.0,-10.10,6.94,3.6,2.0,-0.519,0.698,0.302,41.00,-43.54,0.8,0.7,0.47,0.25,32.30,4.37,0.22,0.04,1.65,0,True,2026-03-28
3,Dylan Cardwell,C,Sacramento Kings,Atlanta Hawks,AWAY,14.5,236.5,player_points,4.5,-102,-103,0.505,0.507,6.7,6.0,NaN,NaN,NaN,4.19,2.2,1.5,-0.525,0.700,0.300,38.63,-40.87,0.8,0.8,0.73,0.54,22.15,6.75,0.13,0.06,1.65,1,True,2026-03-28
4,VJ Edgecombe,SG,Philadelphia 76ers,Charlotte Hornets,AWAY,6.5,230.5,player_points,14.5,-104,-112,0.510,0.528,19.8,19.5,12.00,2.0,-2.50,10.54,5.3,5.0,-0.503,0.693,0.307,35.93,-41.89,0.8,0.7,0.73,0.51,33.36,7.27,0.25,0.04,1.05,1,True,2026-03-28
5,Precious Achiuwa,PF,Sacramento Kings,Atlanta Hawks,AWAY,14.5,236.5,player_points,12.5,-107,100,0.517,0.500,16.1,14.0,9.20,5.0,-3.30,6.98,3.6,1.5,-0.516,0.697,0.303,34.84,-39.40,0.6,0.7,0.67,0.21,32.98,6.44,0.18,0.04,1.65,1,True,2026-03-28
6,Tre Jones,PG,Chicago Bulls,Memphis Grizzlies,AWAY,-4.5,246.0,player_points,14.5,100,-122,0.500,0.550,16.7,17.5,8.33,3.0,-6.17,4.95,2.2,3.0,-0.444,0.671,0.329,34.20,-40.13,0.8,0.8,0.60,0.29,29.39,2.99,0.20,0.03,2.60,0,True,2026-03-29
7,Grayson Allen,SF,Phoenix Suns,Utah Jazz,HOME,-16.5,231.5,player_points,15.5,-106,-111,0.515,0.526,18.1,16.5,15.83,6.0,0.33,5.97,2.6,1.0,-0.436,0.669,0.331,30.01,-37.08,0.2,0.5,0.60,0.35,27.61,4.05,0.27,0.09,1.15,0,True,2026-03-29
9,Devin Booker,SG,Phoenix Suns,Utah Jazz,HOME,-16.5,231.5,player_points,25.5,-112,100,0.528,0.500,28.8,28.5,36.00,6.0,10.50,8.75,3.3,3.0,-0.377,0.647,0.353,22.47,-29.40,0.2,0.6,0.53,0.50,34.54,1.51,0.33,0.07,1.15,0,True,2026-03-29
10,Nique Clifford,SF,Sacramento Kings,Atlanta Hawks,AWAY,14.5,236.5,player_points,10.5,-105,-110,0.512,0.524,12.4,10.5,9.00,1.0,-1.50,5.91,1.9,0.0,-0.321,0.626,0.374,22.22,-28.60,0.6,0.5,0.53,0.28,33.71,7.06,0.16,0.04,1.65,1,True,2026-03-28


In [11]:
rename_map = {
    'PLAYER_NAME': 'Player',
    'POSITION': 'Position',
    'CATEGORY': 'Prop',
    'LINE': 'Line',
    'OPPONENT': 'Opponent',
    'TEAM_SPREAD': 'Spread',
    'GAME_TOTAL': 'Total',
    'OPP_DEF_RATING': 'Opp Def Rating',
    'OPP_RANK_DEF_RATING': 'Opp Def Rank',
    'OPP_PACE': 'Opp Pace',
    'OPP_PACE_RANK': 'Opp Pace Rank',
    'ODDS_OVER': 'Odds Over',
    'ODDS_UNDER': 'Odds Under',
    'IMP_PROB_OVER': 'Implied Over',
    'IMP_PROB_UNDER': 'Implied Under',
    'AVG_STAT_L10': 'Avg Stat L10',
    'MED_STAT_L10': 'Med Stat L10',
    'STD_STAT_L10': 'Std Stat L10',
    'EDGE': 'Edge',
    'MED_EDGE': 'Med Edge',
    'Z_SCORE': 'Z Score',
    'PROB_OVER': 'Prob Over',
    'PROB_UNDER': 'Prob Under',
    'EV_OVER': 'EV Over',
    'EV_UNDER': 'EV Under',
    'OVER_RATE_L5': 'OVER L5',
    'OVER_RATE_L10': 'OVER L10',
    'OVER_RATE_L15': 'OVER L15',
    'OVER_RATE_SEASON': 'ALL SEASON',
    'AVG_MIN_L10': 'Avg Min L10',
    'STD_MIN_L10': 'Std Min L10',
    'AVG_USG_L10': 'Avg USG% L10',
    'STD_USG_L10': 'Std USG% L10',
    'MIN_CONSISTENCY': 'Min Consistency',
    'IS_UNDERDOG': 'Underdog',
    'AVG_STAT_VS_MATCHUP': 'Avg Stat vs Matchup',
    'MATCHUP_GAMES': 'Matchup Games',
}

df = final.rename(columns=rename_map)
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['Prop'] = df['Prop'].map(prop_label_map).fillna(df['Prop'])

df = df[[
    'Player',
    'Position',
    'Prop',
    'Line',
    'Opponent',
    'Odds Over',
    'Odds Under',
    'Implied Over',
    'Implied Under',
    'EV Over',
    'EV Under',
    'Avg Stat L10',
    'Med Stat L10',
    'Std Stat L10',
    'Z Score',
    'Prob Over',
    'Prob Under',
    'OVER L5',
    'OVER L10',
    'OVER L15',
    'Avg Min L10',
    'Std Min L10',
    'Avg USG% L10',
    'Std USG% L10',
    'Avg Stat vs Matchup',
    'Matchup Games',
    'Spread',
    'Total',
    'Opp Def Rating',
    'Opp Def Rank',
    'Opp Pace',
    'Opp Pace Rank',
]].sort_values(by='EV Over', ascending=False)
df.head(10)

,Player,Position,Prop,Line,Opponent,Odds Over,Odds Under,Implied Over,Implied Under,EV Over,EV Under,Avg Stat L10,Med Stat L10,Std Stat L10,Z Score,Prob Over,Prob Under,OVER L5,OVER L10,OVER L15,Avg Min L10,Std Min L10,Avg USG% L10,Std USG% L10,Avg Stat vs Matchup,Matchup Games,Spread,Total,Opp Def Rating,Opp Def Rank,Opp Pace,Opp Pace Rank
0,Joel Embiid,C,PTS,26.5,Charlotte Hornets,120,-160,0.455,0.615,86.34,-75.14,32.1,32.5,5.47,-1.024,0.847,0.153,0.6,0.8,0.73,34.73,5.58,0.35,0.05,27.00,2.0,6.5,230.5,113.6,13,97.90,26
103,Joel Embiid,C,PTS+REB,32.5,Charlotte Hornets,100,-115,0.500,0.535,77.80,-79.25,41.0,42.0,6.96,-1.221,0.889,0.111,0.8,0.9,0.80,34.73,5.58,0.35,0.05,30.50,2.0,6.5,230.5,113.6,13,97.90,26
64,Joel Embiid,C,PTS+REB+AST,36.5,Charlotte Hornets,-105,-113,0.512,0.531,73.76,-79.27,46.4,47.0,8.06,-1.228,0.890,0.110,0.8,0.9,0.80,34.73,5.58,0.35,0.05,37.00,2.0,6.5,230.5,113.6,13,97.90,26
129,Joel Embiid,C,PTS+AST,29.5,Charlotte Hornets,-110,-112,0.524,0.528,70.10,-79.37,37.5,39.0,6.50,-1.231,0.891,0.109,0.8,0.9,0.80,34.73,5.58,0.35,0.05,33.50,2.0,6.5,230.5,113.6,13,97.90,26
1,Tyrese Maxey,PG,PTS,24.5,Charlotte Hornets,-110,-105,0.524,0.512,61.89,-70.32,29.6,29.5,4.95,-1.030,0.848,0.152,0.8,0.9,0.87,35.82,4.86,0.33,0.04,23.60,5.0,6.5,230.5,113.6,13,97.90,26
104,Tyrese Maxey,PG,PTS+REB,27.5,Charlotte Hornets,-125,-105,0.556,0.512,59.84,-78.13,33.5,32.5,4.93,-1.217,0.888,0.112,0.8,0.9,0.80,35.82,4.86,0.33,0.04,27.60,5.0,6.5,230.5,113.6,13,97.90,26
36,Dylan Cardwell,C,REB,6.5,Atlanta Hawks,-105,-108,0.512,0.519,57.17,-62.44,9.4,10.5,3.37,-0.861,0.805,0.195,0.8,0.8,0.73,22.15,6.75,0.13,0.06,NaN,NaN,14.5,236.5,113.1,11,102.59,4
65,Dylan Cardwell,C,PTS+REB+AST,12.5,Atlanta Hawks,-108,-105,0.519,0.512,48.10,-54.90,18.0,18.0,7.47,-0.736,0.769,0.231,0.6,0.7,0.67,22.15,6.75,0.13,0.06,NaN,NaN,14.5,236.5,113.1,11,102.59,4
41,Paul George,SF,AST,3.5,Charlotte Hornets,-102,-115,0.505,0.535,41.60,-46.72,4.2,4.5,1.23,-0.569,0.715,0.285,0.8,0.7,0.60,32.48,2.54,0.21,0.06,8.25,4.0,6.5,230.5,113.6,13,97.90,26
2,Nickeil Alexander-Walker,SG,PTS,20.5,Sacramento Kings,102,-115,0.495,0.535,41.00,-43.54,24.1,22.5,6.94,-0.519,0.698,0.302,0.8,0.7,0.47,32.30,4.37,0.22,0.04,10.40,5.0,-14.5,236.5,120.2,28,100.31,17


In [12]:
output_path = f'data/props/ev_analysis/underdog.csv'
# output_path = f'data/props/ev_analysis/prizepicks.csv'
df.to_csv(output_path, index=False)